# ZombAI V2 — Multi-Course Graph-RAG Pipeline Walkthrough

Welcome to **ZombAI V2**. This notebook walks through the complete academic RAG pipeline step-by-step across **three complete IIT Kanpur courses and textbooks**:
1. **MTH441: Regression Analysis & ANOVA** (Prof. Shalabh) — 6 Lectures (429 Chunks, 41 Equations)
2. **CS771: Introduction to Machine Learning** (Prof. Piyush Rai) — 17 Lectures (384 Chunks, 18 Equations)
3. **EE602: Statistical Signal Processing / Estimation Theory** (Steven M. Kay) — 15 Chapters (1,374 Chunks, 20 Equations)

---

## Curriculum Scope: Multi-Course Architecture

| Course | Instructor / Source | Content Type | Units | Chroma Chunks | Discrete Equations | Knowledge Graph |
|---|---|---|---|---|---|---|
| **MTH441** | Prof. Shalabh | Lecture Slides (PDF) | 6 Lectures | 429 | 41 LaTeX | `MTH441_graph.json` |
| **CS771** | Prof. Piyush Rai | Lecture Slides (PDF) | 17 Lectures | 384 | 18 LaTeX | `CS771_graph.json` |
| **EE602** | Steven M. Kay / EE Faculty | Textbook (PDF) | 15 Chapters | 1,374 | 20 LaTeX | `EE602_graph.json` |
| **Total** | **IIT Kanpur System** | **Multi-Course System**| **38 Units** | **2,187 Chunks** | **79 Equations** | **3 Course Graphs** |


## 0. Environment Setup & Pipeline Imports
Import all modular pipeline components and configure UTF-8 compatibility.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path

# Ensure project root is on sys.path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Configure UTF-8 stdout encoding for math & Greek symbols on Windows
if sys.platform == "win32" and hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# Clear any stale ChromaDB singleton cache in this notebook process
try:
    from chromadb.api.client import SharedSystemClient
    SharedSystemClient.clear_system_cache()
except Exception:
    pass

# Import all modular pipeline components
from src.extractor import extract_pdf
from src.cleaner import clean_pages, clean_page_text
from src.chunker import chunk_pages
from src.embedder import Embedder
from src.vector_store import VectorStore
from src.equation_parser import EquationStore, Equation
from src.graph_store import CourseGraph
from src.generator import LLMGenerator
from src.graph_retriever import GraphRetriever
from src.ingest_manager import IngestManager

print("✓ All ZombAI V2 modules successfully imported!")
print(f"Python interpreter: {sys.executable}")


## 1. Module 1: `src/extractor.py` (Extracting All 6 Lecture PDFs)

**Why this module is called:**
- Reads raw PDF streams page-by-page using **PyMuPDF** (`pymupdf`).
- Normalizes mathematical symbols and preserves 1-based page numbers.
- **Processes all 6 lectures** in `data/lectures/MTH441/`.

In [ ]:
lecture_manifest = [
    ("data/lectures/MTH441/lecture01.pdf", "Lecture 01"),
    ("data/lectures/MTH441/lecture02.pdf", "Lecture 02"),
    ("data/lectures/MTH441/lecture03.pdf", "Lecture 03"),
    ("data/lectures/MTH441/lecture04.pdf", "Lecture 04"),
    ("data/lectures/MTH441/lecture05.pdf", "Lecture 05"),
    ("data/lectures/MTH441/lecture06.pdf", "Lecture 06"),
]

all_raw_pages = []
for pdf_path, lec_name in lecture_manifest:
    pages = extract_pdf(pdf_path, course="MTH441", lecture=lec_name)
    all_raw_pages.extend(pages)
    print(f"Extracted {len(pages):2d} pages from {pdf_path} ({lec_name})")

print(f"\n✓ Total pages extracted across all 6 lectures: {len(all_raw_pages)}")

## 2. Module 2: `src/cleaner.py` (Cleaning All 6 Lectures)

**Why this module is called:**
- Strips recurring slide footers (`Regression Analysis and ANOVA | Chapter X | ... | Shalabh, IIT Kanpur`).
- Reconstructs fragmented mathematical equations (fractions, roots, exponents, densities, summations).
- Normalizes bullet indentations and line wrapping so text reads naturally.

In [ ]:
cleaned_pages = clean_pages(all_raw_pages)
print(f"✓ Cleaned all {len(cleaned_pages)} pages across all 6 lectures.")

# Verify footers are stripped across all lectures
footer_remnants = [p for p in cleaned_pages if "Shalabh, IIT Kanpur" in p["text"]]
print(f"Pages containing slide footer remnants: {len(footer_remnants)} (expected: 0)")

## 3. Module 3: `src/chunker.py` (Chunking All 6 Lectures)

**Why this module is called:**
- Splits continuous text into discrete chunks (target 800 chars, 120 char overlap).
- **Strict in-page scoping**: chunk indexing resets per page to prevent boundary cross-contamination.
- **Natural boundary snapping**: chunks cleanly snap to topic headers, paragraphs, or sentence boundaries.

In [ ]:
all_chunks = chunk_pages(cleaned_pages, chunk_size=800, chunk_overlap=120)
print(f"✓ Total chunks generated across all 6 lectures: {len(all_chunks)}\n")

from collections import Counter
chunk_dist = Counter([c["metadata"]["lecture"] for c in all_chunks])
print("--- Chunk Breakdown by Lecture ---")
for lec, count in sorted(chunk_dist.items()):
    print(f"  • {lec}: {count:2d} chunks")

## 4. Module 4: `src/embedder.py` (Embedding All Chunks)

**Why this module is called:**
- Converts text chunks into 384-dimensional dense vectors using `sentence-transformers/all-MiniLM-L6-v2`.
- Loaded once into memory for fast, consistent embedding inference.

In [ ]:
embedder = Embedder()
print(f"Active embedding backend: {embedder.backend}")
print(f"Embedding dimension: {embedder.dimension}")

print(f"Generating embeddings for {len(all_chunks)} chunks across all 6 lectures...")
embeddings = embedder.embed_chunks(all_chunks, show_progress_bar=False)
print(f"✓ Embeddings matrix shape: {embeddings.shape}")

## 5. Module 5: `src/vector_store.py` (ChromaDB Persistence)

**Why this module is called:**
- Persists chunk embeddings and metadata on disk at `storage/chroma`.
- Creates deterministic unique IDs (`MTH441_Lecture05_p4_c1`).
- Indexes all 429 chunks across all 6 lectures in cosine space.

In [ ]:
vector_store = VectorStore(persist_dir="storage/chroma", collection_name="zombai_lecture_chunks_v1")
current_count = vector_store.count()
print(f"✓ Total records in persistent collection: {current_count}")

# Check if MTH441 needs indexing (upsert without resetting to preserve multi-course data)
if current_count < len(all_chunks):
    upserted = vector_store.add_chunks(all_chunks, embeddings)
    print(f"✓ Indexed {upserted} chunks into ChromaDB!")

# Verify stored chunks by course and lecture
from collections import Counter
all_data = vector_store.collection.get()
stored_lecs = Counter([f"{m.get('course', '')} {m.get('lecture', '')}".strip() for m in all_data["metadatas"]])
print("\n--- Stored Chunks in ChromaDB by Course and Lecture ---")
for lec, count in sorted(stored_lecs.items()):
    print(f"  • {lec}: {count:2d} chunks")


## 6. Knowledge Graph & Discrete Equations (`src/graph_store.py` & `src/equation_parser.py`)

**Why these modules are called:**
- **`graph_store.py`**: Connects Course $\to$ Lectures $\to$ Topics $\to$ Chunks $\leftrightarrow$ Equations in a unified heterogeneous graph (514 nodes, 1,006 edges).
- **`equation_parser.py`**: Maintains 41 First-Class Discrete LaTeX Equations so mathematics is never mangled by PDF extraction.

In [ ]:
# Load All Three Course Knowledge Graphs
mth_graph = CourseGraph.load_json('storage/graph/MTH441_graph.json')
cs_graph = CourseGraph.load_json('storage/graph/CS771_graph.json')
ee_graph = CourseGraph.load_json('storage/graph/EE602_graph.json')

for cg, title in [
    (mth_graph, 'MTH441 (Prof. Shalabh)'),
    (cs_graph, 'CS771 (Prof. Piyush Rai)'),
    (ee_graph, 'EE602 (Steven M. Kay / EE Faculty)'),
]:
    print('=' * 65)
    print(f' Knowledge Graph Summary: {cg.course_id} - {title}')
    print('=' * 65)
    summary = cg.summary()
    print(f' Total Nodes: {summary["total_nodes"]} | Total Edges: {summary["total_edges"]}')
    for ntype, count in summary["node_types"].items():
        print(f'   • {ntype.capitalize():<12}: {count}')
    print()

# Sample Discrete Equations across all three courses
from IPython.display import display, Math, Markdown
eq_store = EquationStore()
print(f'\nTotal Discrete LaTeX Equations in Catalog: {len(eq_store.all_equations())}')

sample_mth = ['MTH441_L02_eq_gauss_markov', 'MTH441_L05_eq_cooks_distance', 'MTH441_L06_eq_vif']
sample_cs = ['CS771_L04_eq_info_gain', 'CS771_L10_eq_svm_soft_margin', 'CS771_L15_eq_pca_objective']
sample_ee = ['EE602_CH03_eq_crlb_scalar', 'EE602_CH06_eq_blue_vector', 'EE602_CH13_eq_kalman_filter_state']

print('\n--- Sample EE602 Equations ---')
for eq_id in sample_ee:
    eq = eq_store.get_by_id(eq_id)
    if eq:
        display(Markdown(f'**{eq.name}** (`{eq.id}` | {eq.course} {eq.lecture}, Page {eq.page}):'))
        display(Math(eq.latex))

print('\n--- Sample CS771 Equations ---')
for eq_id in sample_cs:
    eq = eq_store.get_by_id(eq_id)
    if eq:
        display(Markdown(f'**{eq.name}** (`{eq.id}` | {eq.course} {eq.lecture}, Page {eq.page}):'))
        display(Math(eq.latex))

print('\n--- Sample MTH441 Equations ---')
for eq_id in sample_mth:
    eq = eq_store.get_by_id(eq_id)
    if eq:
        display(Markdown(f'**{eq.name}** (`{eq.id}` | {eq.course} {eq.lecture}, Page {eq.page}):'))
        display(Math(eq.latex))


## 7. Hybrid Graph-RAG Retrieval (`src/graph_retriever.py`)

**Why this module is called:**
- Combines dense semantic vector search (ChromaDB) with knowledge graph traversal.
- Automatically discovers matching discrete LaTeX equations and topic hierarchy.
- Let's test queries targeting different lectures across the curriculum:

In [ ]:
# Initialize Multi-Course GraphRetriever with fresh vector store handle
vector_store = VectorStore()
graph_retriever = GraphRetriever(vector_store=vector_store, embedder=embedder, equation_store=eq_store)

# 1. Query EE602 (Cramer-Rao Lower Bound & Fisher Information)
q_ee = 'Explain the Cramer-Rao Lower Bound and Fisher Information Matrix'
res_ee = graph_retriever.retrieve(query=q_ee, top_k=3, course_filter='EE602')

print('=' * 65)
print(f"EE602 Retrieval: '{q_ee}'")
print(f"Units Retrieved: {res_ee['lectures']}")
for i, c in enumerate(res_ee['chunks'], 1):
    m = c['metadata']
    print(f"  #{i} [{m.get('course')} {m.get('lecture')} Page {m.get('page')}] (Cosine Sim: {c['similarity']})")

print('\nLinked Discrete Equations:')
for eq in res_ee['equations']:
    print(f"  • {eq['id']} - {eq['name']}: $${eq['latex']}$$")

# 2. Query CS771 (Support Vector Machines & Hinge Loss)
print('\n' + '=' * 65)
q_cs = 'Explain soft-margin SVM and how the hinge loss handles margin violations'
res_cs = graph_retriever.retrieve(query=q_cs, top_k=3, course_filter='CS771')
print(f"CS771 Retrieval: '{q_cs}'")
print(f"Units Retrieved: {res_cs['lectures']}")
for i, c in enumerate(res_cs['chunks'], 1):
    m = c['metadata']
    print(f"  #{i} [{m.get('course')} {m.get('lecture')} Page {m.get('page')}] (Cosine Sim: {c['similarity']})")

# 3. Query MTH441 (Cook's Distance)
print('\n' + '=' * 65)
q_mth = "How does Cook's distance detect influential observations?"
res_mth = graph_retriever.retrieve(query=q_mth, top_k=3, course_filter='MTH441')
print(f"MTH441 Retrieval: '{q_mth}'")
print(f"Units Retrieved: {res_mth['lectures']}")
for i, c in enumerate(res_mth['chunks'], 1):
    m = c['metadata']
    print(f"  #{i} [{m.get('course')} {m.get('lecture')} Page {m.get('page')}] (Cosine Sim: {c['similarity']})")


## 8. Local LLM Generation with Llama 3.1 8B (`src/generator.py`)

**Why this module is called:**
- Connects directly to local Ollama (`http://localhost:11434`) running `llama3.1:8b` with GPU acceleration.
- Synthesizes strict, citation-backed academic explanations using the retrieved lecture evidence.

In [ ]:
# Grounded Generation Example: CS771 (SVM & Regularization)
q_gen = 'Explain the soft-margin SVM objective function, its connection to hinge loss, and why L2 regularization is applied.'
ans_gen = graph_retriever.ask(query=q_gen, top_k=4, course_filter='CS771')

display(Markdown(f'### Student Question (CS771): *{q_gen}*'))
display(Markdown(ans_gen['answer']))
display(Markdown(f"---\n**Citations**: {', '.join(ans_gen['citations'])}"))
if ans_gen['referenced_equations']:
    display(Markdown(f"**Linked Equations**: {', '.join(ans_gen['referenced_equations'])}"))


---
## 9. Interactive Multi-Lecture Chat Shell (IPyWidgets)

Test any question across **all 6 course lectures** interactively:

In [ ]:
import time
import ipywidgets as widgets
from IPython.display import display, Markdown

course_dropdown = widgets.Dropdown(
    options=[
        ("EE602: Statistical Signal Processing (Steven M. Kay)", "EE602"),
        ("CS771: Machine Learning (Prof. Piyush Rai)", "CS771"),
        ("MTH441: Regression Analysis (Prof. Shalabh)", "MTH441"),
        ("All Courses (MTH441 + CS771 + EE602)", None),
    ],
    value="EE602",
    description="Course:",
    layout=widgets.Layout(width="48%")
)

query_box = widgets.Text(
    value="Explain the Cramer-Rao Lower Bound and the regularity condition in estimation theory",
    placeholder="Ask an academic question across MTH441, CS771, or EE602...",
    description="Question:",
    layout=widgets.Layout(width="65%")
)

k_slider = widgets.IntSlider(
    value=4,
    min=1,
    max=8,
    step=1,
    description="Top-K:",
    layout=widgets.Layout(width="25%")
)

ask_btn = widgets.Button(
    description="Ask Llama 3.1 8B",
    button_style="primary",
    icon="graduation-cap",
    layout=widgets.Layout(width="20%")
)

clear_btn = widgets.Button(
    description="Clear Chat",
    button_style="warning",
    icon="trash",
    layout=widgets.Layout(width="15%")
)

chat_output = widgets.Output()

# Debounce state to reject queued clicks from WebSocket buffer
_is_currently_querying = False
_last_completed_timestamp = 0.0

def handle_ask_clicked(b):
    global _is_currently_querying, _last_completed_timestamp
    now = time.time()
    
    # Discard click if inference is active OR if within 2.5s cooldown of last finish (drops socket-queued clicks)
    if _is_currently_querying or (now - _last_completed_timestamp < 2.5):
        return
    
    _is_currently_querying = True
    ask_btn.disabled = True
    ask_btn.description = "Thinking (Llama 3.1)..."
    
    # Explicitly clear output widget container
    chat_output.clear_output()
    
    try:
        with chat_output:
            q = query_box.value.strip()
            if not q:
                print("Please enter a question.")
                return
            
            selected_course = course_dropdown.value
            course_label = selected_course or "all indexed courses"
            print(f"Searching {course_label} and querying Llama 3.1 8B for: '{q}'...\n")
            
            res = graph_retriever.ask(query=q, top_k=k_slider.value, course_filter=selected_course)
            
            # Clear the 'Searching...' notice and display clean formatted answer
            chat_output.clear_output()
            print(f"Question: {q} [{course_label}]\n")
            display(Markdown(res["answer"]))
            display(Markdown(f"---\n**Citations**: {', '.join(res['citations'])}"))
            if res["referenced_equations"]:
                display(Markdown(f"**Linked Equations**: {', '.join(res['referenced_equations'])}"))
    except Exception as e:
        with chat_output:
            print(f"Error during generation: {e}")
    finally:
        _last_completed_timestamp = time.time()
        _is_currently_querying = False
        ask_btn.disabled = False
        ask_btn.description = "Ask Llama 3.1 8B"

def handle_clear_clicked(b):
    chat_output.clear_output()

# Clear and register callbacks cleanly
ask_btn._click_handlers.callbacks.clear()
ask_btn.on_click(handle_ask_clicked)

clear_btn._click_handlers.callbacks.clear()
clear_btn.on_click(handle_clear_clicked)

display(widgets.HBox([course_dropdown, k_slider, ask_btn, clear_btn]))
display(query_box)
display(chat_output)
